In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install keras==3.0.5
!pip install keras-cv==0.8.2

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "jax" # you can also use tensorflow or torch

import keras
import keras_cv
import tensorflow as tf # only for data
import tensorflow_io as tfio # for loading .tif files

import cv2
import pandas as pd
import numpy as np
from glob import glob
from tqdm.notebook import tqdm

import matplotlib.pyplot as plt

In [ ]:
print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("KerasCV:", keras_cv.__version__)

In [ ]:
class CFG:
    verbose = 1  # Verbosity
    seed = 42  # Random seed
    preset = "deeplab_v3_plus_resnet50_pascalvoc"  
#     image_size = [384, 384]  # Input image size deeplabv3plus
    image_size = [124, 124]  # Input image size
    seg_image_size = [28,28]
    epochs = 25 # Training epochs
    batch_size = 10  # Batch size
    drop_remainder = True  # Drop incomplete batches
    num_classes = 1 # Number of output classes
    cache = True # Save data into memory during training

In [ ]:
keras.utils.set_random_seed(CFG.seed)

In [ ]:
BASE_PATH = "/kaggle/input/blood-vessel-segmentation"

In [ ]:
mask_paths = sorted(glob(f"{BASE_PATH}/train/*/labels/*tif"))
df = pd.DataFrame({"mask_path":mask_paths})
df['dataset'] = df.mask_path.map(lambda x: x.split('/')[-3])
df['slice'] = df.mask_path.map(lambda x: x.split('/')[-1].replace(".tif",""))

df = df[~df.dataset.str.contains("kidney_3_sparse")]
df['image_path'] = df.mask_path.str.replace("label","image")
df['image_path'] = df.image_path.str.replace("kidney_3_dense","kidney_3_sparse")
df.head()

In [ ]:
df['dataset'].unique()

In [ ]:
CHANNELS = 3 
STRIDE = 3 
for i in range(CHANNELS):
    df[f'image_path_{i:02}'] = df.groupby(['dataset'])['image_path'].shift(-i*STRIDE).ffill()
df['image_paths'] = df[[f'image_path_{i:02d}' for i in range(CHANNELS)]].values.tolist()
df.image_paths[0]

In [ ]:
def build_decoder(with_labels=True, target_size=CFG.image_size, augment=False):
    def decode_image(paths):
        img_array = tf.TensorArray(dtype=tf.uint8, size=len(paths))
        for i in range(len(paths)):
            file_bytes = tf.io.read_file(paths[i])
            img0 = tfio.experimental.image.decode_tiff(file_bytes)[..., 0:1]
            img_array = img_array.write(i, img0[...,0])
        img = tf.transpose(img_array.stack(), perm=(1, 2, 0))
        img = tf.cast(img, tf.float32)
        img -= tf.reduce_min(img)
        img /= tf.reduce_max(img) + 0.001
        del img_array
        return img
    
    def decode_mask(mask_path):
        file_bytes = tf.io.read_file(mask_path)
        msk = tfio.experimental.image.decode_tiff(file_bytes)[...,0:1]
        msk = tf.cast(msk, tf.float32) / 255.0
        return msk

    def decode_without_labels(img_path):
        img = decode_image(img_path)
        img = tf.reshape(img, [*target_size, 3])
        return img
    
    def decode_with_labels(img_path, msk_path):
        img_msk = tf.concat([decode_image(img_path), decode_mask(msk_path)], axis=-1)
        img_msk = tf.image.random_crop(img_msk, [*target_size, 4])
        if augment:
            img_msk = apply_augmentations(img_msk)
        img = tf.reshape(img_msk[...,0:3], [*target_size, 3])
        msk = tf.reshape(img_msk[...,3:4], [*target_size, 1])
        return (img, msk)
    
    def apply_augmentations(img):
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_flip_up_down(img)
        img = tf.image.rot90(img, k=np.random.randint(-3, 3))
        return img
    
    return decode_with_labels if with_labels else decode_without_labels


def build_dataset(img_paths, msk_paths=None, batch_size=32, cache=True,
                  decode_fn=None, augment_fn=None,
                  augment=True, repeat=True, shuffle=1024, 
                  cache_dir="", drop_remainder=False):
    if cache_dir != "" and cache is True:
        os.makedirs(cache_dir, exist_ok=True)
    
    if decode_fn is None:
        decode_fn = build_decoder(msk_paths is not None, augment=augment)
    
    AUTO = tf.data.experimental.AUTOTUNE
    slices = img_paths if msk_paths is None else (img_paths, msk_paths)
    
    ds = tf.data.Dataset.from_tensor_slices(slices)
    ds = ds.map(decode_fn, num_parallel_calls=AUTO)
    ds = ds.cache(cache_dir) if cache else ds
    ds = ds.repeat() if repeat else ds
    if shuffle: 
        ds = ds.shuffle(shuffle, seed=CFG.seed)
        opt = tf.data.Options()
        opt.experimental_deterministic = False
        ds = ds.with_options(opt)
    ds = ds.batch(batch_size, drop_remainder=drop_remainder)
    ds = ds.prefetch(AUTO)
    return ds

In [ ]:
train_df = df[~df.dataset.str.contains('kidney_3')]
valid_df = df[df.dataset.str.contains('kidney_3')]
print('Num Train:', len(train_df), '| Num Valid:', len(valid_df))

In [ ]:
valid_df['dataset'].unique()

In [ ]:
train_image_paths = train_df.image_paths.tolist()
train_mask_paths = train_df.mask_path.tolist()
train_ds = build_dataset(train_image_paths, train_mask_paths, batch_size=CFG.batch_size,
                         cache=CFG.cache, augment=True)

valid_image_paths = valid_df.image_paths.tolist()
valid_mask_paths = valid_df.mask_path.tolist()
valid_ds = build_dataset(valid_image_paths, valid_mask_paths, batch_size=CFG.batch_size,
                         cache=CFG.cache, repeat=False, shuffle=False, augment=False)

In [ ]:
import matplotlib.pyplot as plt
batch = train_ds.take(1).get_single_element()
fig = keras_cv.visualization.plot_segmentation_mask_gallery(
    batch[0],
    value_range=(0, 1),
    num_classes=2,
    y_true=batch[1],
    scale=3,
    rows=2,
    cols=3,
)
print(type(fig))

In [ ]:
plt.figure(figsize=(9, 6))  # Adjust figsize as needed
keras_cv.visualization.plot_segmentation_mask_gallery(
    batch[0],
    value_range=(0, 1),
    num_classes=2,
    y_true=batch[1],
    scale=3,
    rows=2,
    cols=3,
)

# Save the figure as an image file
plt.savefig('segmentation_mask_gallery.png')
# plt.close()

In [ ]:
from keras import ops

class DiceLoss(keras.losses.Loss):
    def __init__(self, smooth=1e-4, name="dice_loss"):
        super().__init__(name=name)
        self.smooth = smooth

    def call(self, y_true, y_pred):
        # Flatten label and prediction tensors
        y_true = ops.ravel(y_true)
        y_pred = ops.ravel(y_pred)
        
        intersection = ops.sum(y_true * y_pred)
        union = ops.sum(y_true) + ops.sum(y_pred)

        dice = (2. * intersection + self.smooth) / (union + self.smooth)
        return 1. - dice
    
class DiceCoef(keras.metrics.Metric):
    def __init__(self, name='dice_coef', smooth=1e-4, threshold=0.5, **kwargs):
        super().__init__(name=name, **kwargs)
        self.smooth = smooth
        self.threshold = threshold
        self.intersection_sum = self.add_weight(name='intersection_sum', initializer='zeros')
        self.union_sum = self.add_weight(name='union_sum', initializer='zeros')

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_pred = ops.cast(y_pred > self.threshold, dtype="float32")
        y_true = ops.ravel(y_true)
        y_pred = ops.ravel(y_pred)

        intersection = ops.sum(y_true * y_pred)
        union = ops.sum(y_true) + ops.sum(y_pred)

        self.intersection_sum.assign_add(intersection)
        self.union_sum.assign_add(union)

    def result(self):
        dice = (2 * self.intersection_sum + self.smooth) / (self.union_sum + self.smooth)
        return dice

    def reset_states(self):
        self.intersection_sum.assign(0)
        self.union_sum.assign(0)
        
    def get_config(self):
        config = super().get_config()
        config.update({'smooth': self.smooth, 'threshold': self.threshold})
        return config

In [ ]:
def additional_encoder(x, filters):
    x = keras.layers.Conv2D(filters, 3, padding='same')(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Activation('relu')(x)
    x = keras.layers.Conv2D(filters, 3, padding='same')(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Activation('relu')(x)
    return x

In [ ]:
segmentation_head = keras.Sequential(
    [
        keras.layers.Conv2D(
            filters=64,  # Increased from 32
            kernel_size=3,  # Changed from 1 to 3
            padding="same",
            use_bias=False,
        ),
        keras.layers.BatchNormalization(),
        keras.layers.ReLU(),
        keras.layers.Conv2D(
            filters=32,
            kernel_size=3,
            padding="same",
            use_bias=False,
        ),
        keras.layers.BatchNormalization(),
        keras.layers.ReLU(),
        keras.layers.UpSampling2D(size=(4, 4), interpolation="bilinear"),
        keras.layers.Conv2D(
            filters=CFG.num_classes,
            kernel_size=1,
            use_bias=False,
            padding="same",
            activation="sigmoid",
            dtype="float32",
        ),
    ], name="segmentation_head",
)

In [ ]:
# Load the full DeepLabV3+ model
backbone = keras_cv.models.DeepLabV3Plus.from_preset(
    CFG.preset,
    input_shape=[*CFG.image_size, 3],
)

# Take only layers from backbone before head
neck_layer_name = backbone.layers[-2].name
out = backbone.get_layer(neck_layer_name).output

# Additional encoder path
additional_features = additional_encoder(backbone.input, 64)
additional_features = keras.layers.MaxPooling2D()(additional_features)
additional_features = additional_encoder(additional_features, 128)
additional_features = keras.layers.MaxPooling2D()(additional_features)
additional_features = additional_encoder(additional_features, 256)

# Combine features
out = keras.layers.Concatenate()([out, additional_features])

# Use newly defined head for segmentation
out = segmentation_head(out)

# Create a new model
model = keras.models.Model(inputs=backbone.input, outputs=out)
OPTIMIZER = keras.optimizers.Adam(learning_rate=1e-4)
METRICS = [
    DiceCoef(),
    keras.metrics.BinaryAccuracy(name="accuracy"),
]
LOSS = DiceLoss()  
model.compile(optimizer=OPTIMIZER, loss=LOSS, metrics=METRICS)

# Model Summary
model.summary()

In [ ]:
# Replace the existing model creation code with this
model = build_custom_deeplabv3_plus(input_shape=[*CFG.image_size, 3], num_classes=CFG.num_classes)

# Compile the model
OPTIMIZER = keras.optimizers.Adam(learning_rate=1e-4)
METRICS = [
    DiceCoef(),
    keras.metrics.BinaryAccuracy(name="accuracy"),
]
LOSS = DiceLoss()  
model.compile(optimizer=OPTIMIZER, loss=LOSS, metrics=METRICS)

# Model Summary
model.summary()

In [ ]:
segmentation_head = keras.Sequential(
    [
        keras.layers.Conv2D(
            filters=32,
            kernel_size=1,
            padding="same",
            use_bias=False,
        ),
        keras.layers.BatchNormalization(),
        keras.layers.ReLU(),
        keras.layers.UpSampling2D(size=(4, 4), interpolation="bilinear"),
        keras.layers.Conv2D(
            filters=CFG.num_classes,
            kernel_size=1,
            use_bias=False,
            padding="same",
            activation="sigmoid",
            dtype="float32",
        ),
    ], name="segmentation_head",
)

In [ ]:
backbone = keras_cv.models.DeepLabV3Plus.from_preset(
    CFG.preset,
    input_shape=[*CFG.image_size, 3],
)

# Take only layers from backbone before head
neck_layer_name = backbone.layers[-2].name
out = backbone.get_layer(neck_layer_name).output
print(out)
# Use newly defined head for segmentation
out = segmentation_head(out)

# Create a new model
model = keras.models.Model(inputs=backbone.input, outputs=out)

# Compile the model
OPTIMIZER = keras.optimizers.Adam(learning_rate=2e-4) 
METRICS = [
    DiceCoef(),
    keras.metrics.BinaryAccuracy(name="accuracy"),
]
LOSS = DiceLoss()  
model.compile(optimizer=OPTIMIZER, loss=LOSS, metrics=METRICS)

# Model Sumamry
model.summary()

In [ ]:
import math

def get_lr_callback(batch_size=8, mode='cos', epochs=10, plot=False):
    lr_start, lr_max, lr_min = 5e-5, 3e-5 * batch_size, 1e-5
    lr_ramp_ep, lr_sus_ep, lr_decay = 3, 0, 0.75

    def lrfn(epoch):  # Learning rate update function
        if epoch < lr_ramp_ep: lr = (lr_max - lr_start) / lr_ramp_ep * epoch + lr_start
        elif epoch < lr_ramp_ep + lr_sus_ep: lr = lr_max
        elif mode == 'exp': lr = (lr_max - lr_min) * lr_decay**(epoch - lr_ramp_ep - lr_sus_ep) + lr_min
        elif mode == 'step': lr = lr_max * lr_decay**((epoch - lr_ramp_ep - lr_sus_ep) // 2)
        elif mode == 'cos':
            decay_total_epochs, decay_epoch_index = epochs - lr_ramp_ep - lr_sus_ep + 3, epoch - lr_ramp_ep - lr_sus_ep
            phase = math.pi * decay_epoch_index / decay_total_epochs
            lr = (lr_max - lr_min) * 0.5 * (1 + math.cos(phase)) + lr_min
        return lr

    if plot:  # Plot lr curve if plot is True
        plt.figure(figsize=(10, 5))
        plt.plot(np.arange(epochs), [lrfn(epoch) for epoch in np.arange(epochs)], marker='o')
        plt.xlabel('epoch'); plt.ylabel('lr')
        plt.title('LR Scheduler')
        plt.show()

    return keras.callbacks.LearningRateScheduler(lrfn, verbose=False)  # Create lr callback

In [ ]:
lr_cb = get_lr_callback(CFG.batch_size, plot=True)

In [ ]:
ckpt_cb = keras.callbacks.ModelCheckpoint("best_model.keras",
                                         monitor='val_dice_coef',
                                         save_best_only=True,
                                         save_weights_only=False,
                                         mode='max')

In [ ]:
history = model.fit(
    train_ds, 
    epochs=CFG.epochs,
    callbacks=[lr_cb, ckpt_cb], 
    steps_per_epoch=len(train_df)//CFG.batch_size,
    validation_data=valid_ds, 
    verbose=CFG.verbose
)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 6)) 
plt.plot(history.history['val_loss'], label='Validation')
plt.title('Validation Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend()

# Save the figure
plt.savefig('val_loss.png', bbox_inches='tight')  # Ensure the entire plot is saved
plt.close()  # Close the plot to free up memory

In [ ]:
import json
with open('training_history.json', 'w') as json_file:
    json.dump(history.history, json_file)

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(9, 6))
plt.plot(history.history['loss'], label='Training',color= 'Orange')
plt.title('Training Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend()
# plt.show()
plt.savefig('train_loss.png')
plt.close()

In [ ]:
dice_loss = history.history['dice_coef']
val_dice_loss = history.history['val_dice_coef']
plt.figure(figsize=(9, 6))
# Plot DICE loss
plt.plot(dice_loss, label='Train DICE Loss',color = 'Orange')
plt.title('DICE Loss')
plt.xlabel('Epoch')
plt.ylabel('DICE Loss')
plt.legend()
# plt.show()
plt.savefig('train_dice_loss.png')
plt.close()

In [ ]:
plt.figure(figsize=(9, 6))
val_dice_loss = history.history['val_dice_coef']
plt.plot(val_dice_loss, label='Validation DICE Loss',color = 'Blue')
plt.title('Validation DICE Loss')
plt.xlabel('Epoch')
plt.ylabel('DICE Loss')
plt.legend()
# plt.show()
plt.savefig('val_dice_loss.png')
plt.close()

In [ ]:
plt.figure(figsize=(9, 6))
dice_loss = history.history['dice_coef']
val_dice_loss = history.history['val_dice_coef']

plt.plot(dice_loss, label='Train DICE Loss',color = 'Orange')
plt.plot(val_dice_loss, label='Validation DICE Loss',color = 'Blue')
plt.title('DICE Loss')
plt.xlabel('Epoch')
plt.ylabel('DICE Loss')
plt.legend()
# plt.show()
plt.savefig('dice_loss_together.png')
plt.close()

In [ ]:
num_samples = int(len(valid_df) * 0.8)

partial_valid_ds = valid_ds.take(num_samples)

predictions = model.predict(partial_valid_ds)
y_true = []
y_pred = []

for batch in valid_ds:
    y_true_batch = batch[1].numpy().flatten()  
    y_pred_batch = model.predict(batch[0]).flatten() 
    y_true.extend(y_true_batch)
    y_pred.extend(y_pred_batch)

y_true = np.array(y_true)
y_pred = np.array(y_pred)

y_pred_binary = (y_pred > 0.5).astype(int)

TP = np.sum((y_true == 1) & (y_pred_binary == 1))
FN = np.sum((y_true == 1) & (y_pred_binary == 0))
TN = np.sum((y_true == 0) & (y_pred_binary == 0))
FP = np.sum((y_true == 0) & (y_pred_binary == 1))

specificity = TN / (TN + FP)

print("Specificity:", specificity)
sensitivity = TP / (TP + FN)

print("Sensitivity:", sensitivity)

In [ ]:
print("Sensitivity:", sensitivity)

In [ ]:
val_accuracy = history.history['val_accuracy']
val_accuracy

In [ ]:
val_dice = history.history['val_dice_coef']
val_dice

In [ ]:
average_val_accuracy = sum(val_accuracy) / len(val_accuracy)
average_val_accuracy

In [ ]:
average_val_dice = sum(val_dice) / len(val_dice)
average_val_dice

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import seaborn as sns

num_samples = int(len(valid_df) * 0.8)
partial_valid_ds = valid_ds.take(num_samples)

predictions = model.predict(partial_valid_ds)

y_true = []
y_pred = []

for batch in valid_ds:
    y_true_batch = batch[1].numpy().flatten()  
    y_pred_batch = model.predict(batch[0]).flatten() 
    y_true.extend(y_true_batch)
    y_pred.extend(y_pred_batch)

y_true = np.array(y_true)
y_pred = np.array(y_pred)

y_pred_binary = (y_pred > 0.5).astype(int)

conf_matrix = confusion_matrix(y_true, y_pred_binary)
plt.figure(figsize=(9, 6))
# Plot confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["Predicted Negative", "Predicted Positive"],
            yticklabels=["True Negative", "True Positive"])
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
# plt.show()
plt.savefig('confusion_matrix.png')
plt.close()